### Disciplina de Análise de Dados    
### Curso de Gestão de Dados
### Universidade Federal do Piauí
##### Prof. Arlino Magalhães
arlino@ufpi.edu.br | @arlino.magalhaes


# Projetos de análise de dados
---

# Mercado de Criptomoedas

*Você foi contratado como um experiente trader, com sólidos conhecimentos em análise de dados, para apoiar uma empresa de investimentos que deseja atuar no mercado de criptomoedas. Seu primeiro desafio é analisar o comportamento do par BTC-USD (Bitcoin cotado em Dólares americanos). A empresa pretende tomar decisões estratégicas de investimento com base em evidências extraídas de dados históricos, buscando identificar padrões, tendências e possíveis oportunidades de compra e venda. Ao explorar o conjunto de dados, você percebe que há informações relevantes que podem ser utilizadas para compreender a variação de preços ao longo do tempo. Sua missão é realizar uma análise dos dados disponíveis e responder, com base nos dados do par BTC-USD, quais tendências podem ser identificadas e que estratégias de investimento poderiam ser sugeridas para operações futuras.*

Este estudo de caso foi inspirado no trabalho de [Cordeiro et al. (2025)](https://doi.org/10.5753/dsw.2025.247689) cujo código fonte está disponível para [download](https://github.com/jeovanereges/DSW2025).

Instação do pacote da API Finance para recuperar a base de dados.

`pip install yfinance --upgrade --no-cache-dir`

Uma breve explicção dos parametros usados na API:

- `"BTC-USD"` → símbolo do ativo (Bitcoin em dólares).
- `multi_level_index=False` → evita que o DataFrame venha com índice hierárquico.
- `start=start` → data inicial (variável definida anteriormente).
- `end=end` → data final (variável definida anteriormente).
- `interval="1d"` → frequência diária.
- `session=session` → usa uma sessão HTTP para performance/requisições (variável definida anteriormente).

In [1]:
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from curl_cffi import requests

ModuleNotFoundError: No module named 'yfinance'

In [ ]:
session = requests.Session(impersonate="chrome")
start = '2017-01-01' #AAAA-MM-DD
end = '2025-12-31'

dataset = yf.download("BTC-USD",\
                      multi_level_index=False, 
                      start=start, end=end, 
                      interval="1d", 
                      session=session)[["Open", "High", "Low", "Close"]]

## Análise Exploratória

In [ ]:
dataset.info()

In [ ]:
dataset.head()

In [ ]:
dataset.describe()

| Coluna |                          Descrição                          | Tipo    | Valores ausentes |
|--------|:-----------------------------------------------------------:|---------|------------------|
| Open   | Abertura: o primeiro preço  negociado no início do período. | float64 |        não       |
| High   |      Máxima: o maior preço  atingido durante o período.     | float64 |        não       |
| Low    |      Mínimo: o menor preço  atingido durante o período.     | float64 |        não       |
| Close  |  Fechamento: o último preço  negociado ao final do período. | float64 |        não       |

## Tratamento dos dados

Apenas o preço de fechamento é necessário para avaliar a variação do nível dos valores do ativo.

In [ ]:
df_close = dataset['Close']
df_close.head()

A data deve ser movida para a coluna para facilitar a manipulação das datas.

In [ ]:
df_close = df_close.reset_index()
df_close.head()

As colunas Year e Month devem ser criadas para facilitar os cálculos.

In [ ]:
df_close['Year'] = df_close['Date'].apply(lambda time: time.year)
df_close['Month'] = df_close['Date'].apply(lambda time: time.month)
df_close.head()

Apenas o preço de fechamento do mês é necessário para analisar analisar a variação do nível mensal. Por esse motivo, apenas a linha do último dia de cada mês é selecionada.

In [ ]:
df_close = df_close[df_close['Date'].dt.is_month_end]
df_close

## Análise dos dados

Gráfico da variação do valor nível dos preços durante os alunos. O valor cresce com o passar dos anos.

In [ ]:
plt.plot(df_close['Date'], df_close['Close'])
plt.savefig('plot-Date-proj-cripto.png', dpi=300)
plt.show()

Agrupa os dados pelo par ano/mês (Year/Month). O valores selecionados são os de Close. Como há apenas um valor de fechamento para cada par, nenhum calculo é feito, é retornado apenas o valor do fechamento. Ou seja, qualquer agregação daria o mesmo resultado.

In [ ]:
df_close_grouped = df_close.groupby(['Year', 'Month'])['Close'].max()
df_close_grouped.round(2)

Faz o pivoteamento do agrupamento para melhorar a visualização dos dados. Podemos observar que os valores crescem com o passar do tempo.

In [ ]:
df_close_grouped.unstack()

Cria a coluna **pct** que calcula a variação percentual entre os valores consecutivos do preço fechamento (Close).

In [ ]:
df_close['pct'] = (df_close['Close'].pct_change()*100).round(2)
df_close

Gráfico da variação percentual (pct). Esse gráfico não parece útil.

In [ ]:
plt.plot(df_close['Date'], df_close['pct'])
plt.grid(True)
plt.savefig('plot-pct-proj-cripto.png', dpi=300)
plt.show()

Agrupa os dados pelo par ano/mês (Year/Month). O valores selecionados são os de ptc. Como há apenas um valor de pct para cada par, nenhum calculo é feito, é retornado apenas o valor do pct. Ou seja, qualquer agregação daria o mesmo resultado.

Faz o pivoteamento do agrupamento para melhorar a visualização dos dados. Podemos observar que os valores do pct variam de formas diferentes como passar to tempo.

In [ ]:
df_close_grouped = df_close.groupby(['Year', 'Month'])['pct'].max()
df_close_grouped = df_close_grouped.unstack()
df_close_grouped

Estiliza o dataframe para melhor a variação dos valores de pct. Agora é mais fácil observar quando o ativo cresceu ou diminuiu de um mês para outro.

In [ ]:
labels = ['jan', 'fev', 'mar', 'abr', 'mai', 'jun', 'jul', 'ago', 'set',\
                                                      'out', 'nov','dez']
df_close_grouped.columns = labels
df_style = df_close_grouped.style.map(lambda v: 'background: lightcoral' \
                                       if v < 0 else 'background: lightblue')
df_style

Calcula a média dos pct em cada mês do ano.

In [ ]:
mean = df_close_grouped.mean()
mean

Faz o gráfico da média dos pcts em cada mês. Podemos observar que os meses de junho e setembro foram meses de queda seguidos de alta no valor do ativo.

In [ ]:
colors = ['lightcoral' if x <0 else 'lightblue' for x in mean]
plt.bar(mean.index, df_close_grouped.mean(), color=colors)
plt.savefig('bar-mean-pct-proj-cripto.png', dpi=300)

As variações anuais dos valores seguem o padrão que repete frequentemente. Nesse contexto, o mês de junho e, principalmente,setembro são meses indicados para a compra do ativo, visto que ele irá se valorizar no mês seguinte. 